In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
df = pd.read_csv('/kaggle/input/datasets/spscientist/students-performance-in-exams/StudentsPerformance.csv')

print(df.head())
print(df.columns.tolist())
print(df.shape)
print(df.dtypes)

In [ ]:
print(df.isnull().sum())

df.isnull() gives true for any missing values in dataset and false for all filled values. I have printed df.isnull to check that all appear false . Other than that the sum and mean of it also indicate that no data is missing.
However if it was missing , I would fill it with mean or median of the remaining cells.

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
print(df.columns.tolist())

In [ ]:
df['gender'] = df['gender'].str.lower()
print(df['gender'])

We can also use str.strip() to remove whitespaces either in the start or in the end of the data .

In [ ]:
df['gender'] = df['gender'].str.strip()
print(df['gender'])

In [ ]:
df['total_score'] = df['math_score'] + df['reading_score'] + df['writing_score']
top10=df.nlargest(10, 'total_score')
print(top10)

In [ ]:
avgbyedu=df.groupby('parental_level_of_education')['total_score'].mean()
avgbygen=df.groupby('gender')['total_score'].mean()
avgbylun=df.groupby('lunch')['total_score'].mean()

In [ ]:
print(avgbyedu)

In [ ]:
print(avgbygen)

In [ ]:
print(avgbylun)

Observations made from the data :

Parent's education is directly related to their children's scores . The higher their education , more likely their child will perform better.

Females have a higher average than males.

A better lunch has a higher mark average but that might most likely correlate to parent's financial status which will circle back to their educational status . (All talking about averages here ) . It is difficult to deduce anything from lunch type. 

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df['math_score'], bins=40, color='blue', edgecolor='black')
plt.title('Math Score Distribution')
plt.xlabel('Score')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df['reading_score'], bins=40, color='green', edgecolor='black')
plt.title('Reading Score Distribution')
plt.xlabel('Score')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df['writing_score'], bins=40, color='red', edgecolor='black')
plt.title('Writing Score Distribution')
plt.xlabel('Score')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
df.groupby('gender')[['math_score', 'reading_score', 'writing_score']].mean().plot(kind='bar', edgecolor='white')
plt.title('Avg Subject Scores by Gender')
plt.xlabel('Gender')
plt.ylabel('Average Score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
df.groupby('lunch')[['math_score', 'reading_score', 'writing_score']].mean().plot(kind='bar', edgecolor='white')
plt.title('Avg Subject Scores by Lunch Type')
plt.xlabel('Lunch Type')
plt.ylabel('Average Score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
df.groupby('parental_level_of_education')[['math_score', 'reading_score', 'writing_score']].mean().plot(kind='bar', edgecolor='white')
plt.title('Avg Subject Scores by Parental Education')
plt.xlabel('Education Level')
plt.ylabel('Average Score')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))

corr = df[['math_score', 'reading_score', 'writing_score']].corr()

sns.heatmap(corr, annot=True, cmap='coolwarm')

plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

Insights:

Scores are mostly normally distributed with most of the students scoring 60-80 in all three subjects. Maths tend to be more extreme as more students score low and high marks but reading and writing are more balanced towards 70s or higher 70s.

Females , Students with standard lunch and students whose parents are well educated tend to perform well . As previously stated , lunch can act as a proxy for socioeconomic status. Other than that parents who are educated tend to focus more on their child's education . Females tend to score better on reading and writing while males score better in maths .

Subjects are correlated . Reading and writing are very much correlated as they both most likely are language courses . A student being bad or good in one translates to other as well.

In [ ]:
df['total_score'] = df['math_score'] + df['reading_score'] + df['writing_score']
print(df['total_score'].head())

I chose total score and not effort score because study_time or absences could not be found in the dataset.

Why total score helps a model:
total_score combines all three subject scores into one value.
This gives the model a single measure of overall academic performance.

In [ ]:
#1/0 encoding for things that only have two options
df['lunch'] = df['lunch'].map({'standard': 1, 'free/reduced': 0})
df['test_preparation_course'] = df['test_preparation_course'].map({'completed': 1, 'none': 0})
df['gender'] = df['gender'].map({'female': 1,'male': 0})

In [ ]:
# One-hot encoding for columns with more than 2 categories
df = pd.get_dummies(df, columns=['gender', 'race/ethnicity', 'parental_level_of_education'])

In [ ]:
print(df.head())
print(df.columns.tolist())

In [ ]:
scaler = MinMaxScaler()

cols_to_scale = ['math_score', 'reading_score', 'writing_score']

df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

print(df[cols_to_scale].describe().round(2))

ML models require scaling to see every feature uniformly.
For example, if math_score ranges from 0 to 100 and another feature
ranges from 0 to 1, the model will treat math_score as 100x more
important simply because its numbers are larger.
Minmax scaling shrinks every value from 0 to 1.


In [ ]:
df.to_csv('processed.csv', index=False)
print("Saved! Shape:", df.shape)